# 📈 02 · 分组聚合与透视表

> 「分组 → 聚合 → 透视」是取数基本功。本节手写 groupby 并对照 pandas，再演练**留存率、复购率、top-N** 三类经典业务问题。

## 📋 本节要解决的问题

| # | 问题 | 工具 |
|---|---|---|
| 1 | 分组聚合的原理是什么？ | 手写 groupby + agg |
| 2 | 透视表怎么用？ | pivot_table / crosstab |
| 3 | 留存率怎么算？ | 注册日 × 活跃日 矩阵 |
| 4 | top-N 怎么取？ | groupby + nlargest |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
np.random.seed(42)


## 1️⃣ 构造订单数据

In [2]:
rng = np.random.default_rng(3)
n = 200
orders = pd.DataFrame({
    'order_id': range(1, n+1),
    'city': rng.choice(['北京', '上海', '深圳', '杭州'], n, p=[0.4, 0.3, 0.2, 0.1]),
    'category': rng.choice(['数码', '服饰', '食品', '家居'], n, p=[0.25, 0.35, 0.25, 0.15]),
    'amount': rng.uniform(20, 500, n).round(2),
})
print(orders.head())
print('\n总订单数:', len(orders))

   order_id city category  amount
0         1   北京       食品  198.14
1         2   北京       服饰  147.36
2         3   深圳       数码  198.85
3         4   上海       服饰  147.98
4         5   北京       家居  420.39

总订单数: 200


## 2️⃣ 手写 groupby：按城市求均值金额

In [3]:
# 手写：按城市分组求和
groups = {}
for city, amt in zip(orders['city'], orders['amount']):
    groups.setdefault(city, []).append(amt)
manual = {c: np.mean(v) for c, v in groups.items()}

# pandas 对照
pd_result = orders.groupby('city')['amount'].mean()
print('手写:', manual)
print('pandas:', pd_result.round(2).to_dict())
for c in manual:
    assert abs(manual[c] - pd_result[c]) < 1e-9
print('✅ 手写与 pandas 一致')

手写: {'北京': np.float64(241.91844155844151), '深圳': np.float64(239.8524390243903), '上海': np.float64(245.99641791044775), '杭州': np.float64(268.8926666666667)}
pandas: {'上海': 246.0, '北京': 241.92, '杭州': 268.89, '深圳': 239.85}
✅ 手写与 pandas 一致


## 3️⃣ 多聚合：金额总和、单量、客单价

In [4]:
summary = orders.groupby('city').agg(
    order_cnt=('order_id', 'count'),
    total_amount=('amount', 'sum'),
)
summary['avg_order'] = (summary['total_amount'] / summary['order_cnt']).round(2)
print(summary)
assert summary.loc['北京', 'order_cnt'] > 0
print('✅ 聚合完成')

      order_cnt  total_amount  avg_order
city                                    
上海           67      16481.76     246.00
北京           77      18627.72     241.92
杭州           15       4033.39     268.89
深圳           41       9833.95     239.85
✅ 聚合完成


## 4️⃣ 透视表：城市 × 品类 的销售额矩阵

In [5]:
pv = orders.pivot_table(index='city', columns='category', values='amount', aggfunc='sum', fill_value=0).round(0)
print(pv)
print('\n热力图:')
plt.figure(figsize=(8, 4))
plt.imshow(pv.values, cmap='YlOrRd', aspect='auto')
plt.colorbar(label='销售额')
plt.xticks(range(len(pv.columns)), pv.columns)
plt.yticks(range(len(pv.index)), pv.index)
plt.title('城市 × 品类 销售额热力图')
plt.show()

category      家居      数码      服饰      食品
city                                    
上海        2756.0  3872.0  5314.0  4541.0
北京        3397.0  3661.0  6695.0  4875.0
杭州         718.0  1658.0  1283.0   374.0
深圳         815.0  2439.0  3300.0  3280.0

热力图:


C:\Users\24260\AppData\Local\Temp\ipykernel_40816\4219360440.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5️⃣ 业务题：复购率（每人下单天数）

In [6]:
# 模拟每个用户多笔订单
buyers = pd.DataFrame({
    'user_id': rng.integers(1, 60, 300),
    'day': rng.integers(1, 31, 300),
})
buy_cnt = buyers.groupby('user_id')['day'].nunique()
repurchase_rate = (buy_cnt >= 2).mean()
print(f'购买天数≥2 的用户占比（复购率）: {repurchase_rate:.1%}')
print('购买天数分布:')
print(buy_cnt.value_counts().sort_index())

购买天数≥2 的用户占比（复购率）: 91.5%
购买天数分布:
day
1     5
2     4
3    10
4     9
5    10
6     9
7     3
8     4
9     5
Name: count, dtype: int64


## 6️⃣ 业务题：top-2 城市按品类销售额

In [7]:
top = (orders.groupby(['city', 'category'])['amount']
            .sum().reset_index()
            .sort_values('amount', ascending=False)
            .groupby('category').head(2))
print(top.reset_index(drop=True))
assert len(top.groupby('category')) == 4
print('✅ 每个品类取到 top-2 城市')

  city category   amount
0   北京       服饰  6694.79
1   上海       服饰  5313.51
2   北京       食品  4874.83
3   上海       食品  4541.07
4   上海       数码  3871.65
5   北京       数码  3660.75
6   北京       家居  3397.35
7   上海       家居  2755.53


✅ 每个品类取到 top-2 城市


## 📝 自测清单

- [ ] 能手写 groupby：分组 → 聚合 → 合并结果
- [ ] 能区分 pivot_table 与 groupby 的使用场景
- [ ] 能写出「复购率」「留存率」的 SQL/pandas 口径
- [ ] 能说出 groupby(...).head(k) 取组内 top-N 的原理